# Proyecto chatbot para Educacion

Se propone el desarrollo de una prueba de concepto de un chatbot educativo que utilice la metodología RAG (Retrieval Augmented Generation) para responder de manera precisa a preguntas sobre el contenido de un curso universitario. Este chatbot, alimentado con las notas de clase de un curso existente, funcionará como un tutor virtual personalizado, disponible las 24 horas del día para los estudiantes. La iniciativa tiene como objetivo principal mejorar la experiencia de aprendizaje al proporcionar un recurso adicional para consultar el conocimiento contenido en los apuntes elaborados por los docentes. El chatbot será capaz de comprender preguntas complejas, encontrar la información relevante en las notas de clase y generar respuestas coherentes y concisas. Se utilizarán redes neuronales preentrenadas, de acceso abierto y alojadas de forma local en el servidor de la facultad. Para el desarrollo de la prueba de concepto se utilizará el lenguaje Python. El proyecto abarca desde la recopilación y procesamiento de las notas de clase en formato PDF, Word u otro formato similar, hasta el desarrollo de una interfaz conversacional intuitiva mediante la librería Streamlit o similar.
Se espera que al utilizar el chatbot la consulta de los apuntes de clase por parte de los estudiantes aumente, de modo que se logre un aprendizaje más personalizado, una mayor comprensión de los conceptos y un ahorro de tiempo para los estudiantes. Además, se espera que este chatbot sea una herramienta valiosa para los docentes, al proporcionarles información sobre las áreas en las que los estudiantes tienen más interés o dificultades.

## Setup

In [1]:
import os
import numpy as np
import pandas as pd
#Retrieval libraries
from unidecode import unidecode
import re
import torch
from typing import List, Dict, Any
from langchain.schema import Document
from langchain.text_splitter import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings, LlamaCppEmbeddings

from langchain_community.vectorstores import Chroma
#from langchain.retrievers.document_compressors import FlashrankRerank
from langchain.retrievers import ContextualCompressionRetriever
#from langchain_community.document_compressors import SentenceTransformerRerank
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers.document_compressors import EmbeddingsFilter # <-- Importar
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

#LLM libraries
from llama_cpp import Llama
#from transformers import pipeline
#from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.messages import HumanMessage

## Setup Fase Generativa

In [2]:
from dotenv import load_dotenv

def get_api_key():
    """
    Loads the LLM API key from the .env file and returns it.
    
    Raises:
        ValueError: If the LLM_API_KEY is not found in the environment.
        
    Returns:
        str: The LLM API key.
    """
    # This line loads the environment variables from the .env file
    load_dotenv()
    
    # os.getenv() retrieves the value of the environment variable
    api_key = os.getenv("GOOGLE_API_KEY")
    
    # This is a critical security and robustness check
    if not api_key:
        raise ValueError("API Key not found. Make sure you have a .env file with LLM_API_KEY defined.")
        
    return api_key

#####################################
try:
    my_llm_key = get_api_key()
        
        # Now you can use the key in your application
    print("Successfully loaded API Key.")
        # For security, we only show the first and last few characters
    print(f"Key starts with: {my_llm_key[:4]}... and ends with: ...{my_llm_key[-4:]}")
        
        # Example of using the key with a fictional API call
        # some_llm_library.authenticate(api_key=my_llm_key)
        
except ValueError as e:
    print(f"Error: {e}")
        
os.environ["GOOGLE_API_KEY"] = my_llm_key
######################################

Successfully loaded API Key.
Key starts with: AIza... and ends with: ...CoTw


In [3]:
def LoadGoogleLLM():
    """Instantiates and returns a Google LLM through the LangChain interface."""
    # Ensure your GOOGLE_API_KEY is set as an environment variable
    llm = ChatGoogleGenerativeAI(model="gemma-3-12b-it", temperature=0)
    return llm

def LoadLLMFromOllama():
    # Obtener la URL del servicio Ollama de una variable de entorno
    # Por defecto, Ollama escucha en el puerto 11434
    #ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434") # Asumiendo 'ollama' como nombre de servicio en Podman Compose
    ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://192.168.54.106:11434")
    #ollama_model = os.getenv("OLLAMA_MODEL", "llama3.2:3b-instruct-q4_0") # Modelo por defecto, si no se especifica
    ollama_model = os.getenv("OLLAMA_MODEL", "llama3.2:3b-instruct-q8_0") # Modelo por defecto, si no se especifica
    #qwen3:0.6b
    #gemma3:1b
    #gemma3n:e2b
    #llama3.2:3b-instruct-q4_0
    #qwen3:4b-instruct-2507-q4_K_M
    st.info(f"Conectando a Ollama en {ollama_base_url} con el modelo {ollama_model}...") # NUEVO: Para feedback en UI

    try:
        llm = Ollama(base_url=ollama_base_url, model=ollama_model,temperature=0) #IMPORTANTE, usar 0 para RAG.
        # Puedes intentar una pequeña consulta para verificar la conexión
        # llm.invoke("Hola")
        return llm
    except Exception as e:
        st.error(f"No se pudo conectar a Ollama: {e}. Asegúrate de que el servicio Ollama esté corriendo y el modelo '{ollama_model}' esté descargado.")
        st.stop() # Detiene la ejecución de Streamlit si hay un error crítico


def LoadLLMFromllamaCPP():
    #model_path = "/home/nico/.cache/llama.cpp/unsloth_gemma-3-1b-it-GGUF_gemma-3-1b-it-Q4_K_M.gguf"
    model_path = "/home/nico/.cache/llama.cpp/gemma-3-1b-pt-q4_0.gguf"
    
    try:
        llm = Llama(
            model_path=model_path,
            #n_gpu_layers=0, 
            n_ctx=2048, # Context window size
            #temperature=0.1,
            verbose=False
            )
        return llm
    except Exception as e:
        print(f"No se pudo conectar al modelo en llama.cpp: {e}.")
        
def LoadLocalGemma():
    #Instantiate a local LLM
    llm = HuggingFacePipeline.from_model_id(
        model_id="Google/Gemma-3-1b-it", 
        device=-1,
        task="text-generation",
        pipeline_kwargs={"max_new_tokens": 500})
    return llm

In [4]:
#llm=LoadLLMFromllamaCPP()
llm=LoadGoogleLLM()

## Setup Fase Retrieval

In [5]:
# --- Configuration ---
# Chequeo dinámico de dispositivo (CPU o GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

# Use a specific folder to store the database, models, etc.
PERSIST_DIRECTORY = "./db_chroma"
MODEL_CACHE_DIR = "./model_cache"
SOURCE_DOCS_PATH = "./knowledgeBase" 

GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/bge-m3-q8_0.gguf"
#https://huggingface.co/BAAI/bge-m3
#GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/embeddinggemma-300m-Q4_0.gguf"

# This is now the correct way to create the embedding encoder
embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=4096,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
#embedding_encoder = LlamaCppEmbeddings(model_path="/path/to/model/ggml-model-q4_0.bin",n_ctx=4096,n_gpu_layers=0)
#llm = Llama.from_pretrained(
#    repo_id="KimChen/bge-m3-GGUF",
#    filename="bge-m3-f32.gguf",
#)


# Define los parámetros de tu modelo de embedding
#model_name = 'BAAI/bge-m3'
# Multi-Functionality: It can simultaneously perform the three common retrieval functionalities of embedding model: dense retrieval, multi-vector retrieval, and sparse retrieval.
# Multi-Linguality: It can support more than 100 working languages.
# Multi-Granularity: It is able to process inputs of different granularities, spanning from short sentences to long documents of up to 8192 tokens.
#model_kwargs = {'device': device}
#encode_kwargs = {'normalize_embeddings': True}
# Crea la instancia del embedding encoder
#embedding_encoder = HuggingFaceEmbeddings(
#    model_name=model_name,
#    model_kwargs=model_kwargs,
#    encode_kwargs=encode_kwargs,
#    cache_folder=MODEL_CACHE_DIR
#)


retrieval_reranker=HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3"
    #"BAAI/bge-reranker-v2-m3"
    #"BAAI/bge-reranker-v2-minicpm-layerwise"
    #"Alibaba-NLP/gte-multilingual-reranker-base" #General Text Embedding"
)

Usando dispositivo: cpu


llama_context: n_ctx_per_seq (4096) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


## Funciones auxiliares

In [6]:
def process_markdown_document(file_path):
    """Loads a Markdown document and returns a LangChain Document object.

    Args:
        file_path (str): The path to the Markdown file.

    Returns:
        langchain.document_loaders.TextLoader: A TextLoader object containing the document.
    """

    loader = TextLoader(file_path,encoding="UTF8")
    documents = loader.load()

    # Split documents by Markdown sections
    headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4"),
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=True)
    md_header_splits = markdown_splitter.split_text(documents[0].page_content)
    return md_header_splits

# Helper function for printing docs
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )
    
def EmbeddDocsAndPersist(all_splits,embedding_encoder,PERSIST_DIRECTORY):
    # Crear embeddings y persistir la DB en disco
    # Esto solo se hace una vez o cuando los documentos cambian
    print("Creando y persistiendo la base de datos de vectores...")
    vectorStore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding_encoder,
        persist_directory=PERSIST_DIRECTORY
    )
    print("Base de datos creada y guardada.")
    return vectorStore

def load_persisted_db(embedding_encoder,PERSIST_DIRECTORY):
    # Cargar la DB desde el disco
    print("Cargando base de datos persistente...")
    vectorStore = Chroma(
        persist_directory=PERSIST_DIRECTORY,
        embedding_function=embedding_encoder
    )
    print("Base de datos cargada.")
    return vectorStore

#Define a function for joining retrieved chunks
def join_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

## Inicilizacion de base de datos

In [7]:
all_docs = [] # <--- Lista para acumular todos los documentos

for filename in os.listdir(SOURCE_DOCS_PATH): # <--- Iteramos sobre los archivos
    if filename.endswith(".md"): # <--- Filtramos solo archivos .md
        file_path = os.path.join(SOURCE_DOCS_PATH, filename) # <--- Construimos la ruta completa
        processed_document_chunks = process_markdown_document(file_path) # <--- Procesamos cada archivo
        all_docs.extend(processed_document_chunks) # <--- Agregamos los documentos
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100,
    separators=["\n\n"])

chunked_splits = text_splitter.split_documents(all_docs)

# Add metadata
for i, doc in enumerate(chunked_splits):
    doc.metadata["doc_id"] = f"chunk_{i}"

In [8]:
chunked_splits

[Document(metadata={'Header 1': '1 INTRODUCCIÓN AL PROBLEMA', 'doc_id': 'chunk_0'}, page_content='Un problema de **Valores y Vectores Propios** consiste en encontrar los vectores \\(v\\) tales que son direcciones invariantes de la transformación lineal dada por la matriz \\(A\\) (NxN). Esto se expresa como: \\(A v = \\lambda v \\) siendo \\(\\lambda\\) el escalar que cambia el módulo del vector cuya dirección permanece invariante. Se denomina **autovalor** \\( \\lambda \\) y **autovector** \\(v\\). El sistema de ecuaciones puede escribirse de la forma: \\((A - \\lambda I) v = 0 \\) donde interesan las soluciones \\(v\\) distintas de la trivial,\\(v = 0\\). Esto está garantizado sí y sólo sí \\(det(A - \\lambda I) = 0\\). El determinante constituye un polinomio de grado N en el autovalor \\(\\lambda \\), y se denomina **polinomio característico**. Las raíces de dicho polinomio son los autovalores \\(\\lambda \\) de la matriz \\(A\\) para los cuales existen los autovectores o direcciones

In [9]:
#Codificacion de los chunks y Creacion de la base de datos
if len(os.listdir(PERSIST_DIRECTORY))==0:
    vectorStore = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY)
else:
    vectorStore = load_persisted_db(embedding_encoder,PERSIST_DIRECTORY)

Creando y persistiendo la base de datos de vectores...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

Base de datos creada y guardada.


In [20]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/bge-m3-q8_0.gguf"
PERSIST_DIRECTORY = "./db_chroma/bge_m3"
#https://huggingface.co/BAAI/bge-m3
#GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/embeddinggemma-300m-Q4_0.gguf"

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=8192,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
vectorStore_bge_m3_q8 = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Creando y persistiendo la base de datos de vectores...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

Base de datos creada y guardada.


In [21]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/nomic-embed-text-v1.5.Q8_0.gguf"
PERSIST_DIRECTORY = "./db_chroma/nomic"

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=2048,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
vectorStore_nomic = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Creando y persistiendo la base de datos de vectores...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

Base de datos creada y guardada.


In [23]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/all-MiniLM-L6-v2-Q8_0.gguf"
PERSIST_DIRECTORY = "./db_chroma/allminillm6"

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=512,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
vectorStore_allminilm = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Creando y persistiendo la base de datos de vectores...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

Base de datos creada y guardada.


In [38]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/nomic-embed-text-v2-moe.Q6_K.gguf"
PERSIST_DIRECTORY = "./db_chroma/nomic_moe"

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=512,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
vectorStore_nomic_moe = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Creando y persistiendo la base de datos de vectores...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

Base de datos creada y guardada.


In [42]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/Qwen3-Embedding-0.6B-Q8_0.gguf"
PERSIST_DIRECTORY = "./db_chroma/qwen"

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=32768,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
vectorStore_qwen = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY)

llama_context: n_ctx_per_seq (512) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
init: embeddings required but some input tokens were not marked as outputs -> overriding


Creando y persistiendo la base de datos de vectores...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

KeyboardInterrupt: 

In [44]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/mxbai-embed-large-v1.Q8_0.gguf"
PERSIST_DIRECTORY = "./db_chroma/mxbai"

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=512,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
vectorStore_mxbai = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Creando y persistiendo la base de datos de vectores...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

Base de datos creada y guardada.


## Testing Fase Retrieval

### Armado de dataset de validacion

> Usamos IA generativa externa para crear casos de validacion a partir de los chunks. 

In [10]:
system_instructions = ""
user_prompt= f"Resume el siguiente texto en un parrafo. TEXTO: {chunked_splits[0].page_content}"
full_prompt_string = (
        f"{system_instructions}\n\n"
        "--- \n\n" 
        f"{user_prompt}"
    )
messages = [HumanMessage(content=full_prompt_string)]
model_output = llm.invoke(messages)
print("--- Model Output Object ---")
print(model_output)
print("\n--- Final Result (Content) ---")
final_result = model_output.content.strip()
print(final_result)

--- Model Output Object ---
content='El problema de valores y vectores propios busca identificar las direcciones invariantes (autovectores) de una transformación lineal representada por una matriz A, y el factor de escala (autovalor) que modifica la magnitud de estos vectores al aplicar la transformación. Para encontrar estos valores y vectores, se resuelve un sistema de ecuaciones derivado del polinomio característico, que es un polinomio que depende del autovalor. Existen diferentes métodos para lograrlo, desde la resolución directa del polinomio (útil para matrices pequeñas) hasta métodos de transformación que diagonalizan la matriz, y métodos iterativos como el Método de la Potencia, que aproximan los valores y vectores propios de forma sucesiva.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemma-3-12b-it', 'safety_ratings': []} id='run--85ab7fe6-d457-4b6b-93ea-a5f90eb34989-0' usage_me

In [11]:
max_tokens = 150
temperature = 0
top_p = 0.05
echo = False
stop = ["\n\n"]

evaluation_dataset_AIgenerated=[]

for chunk in  chunked_splits:
    chunkContent=chunk.page_content
    chunkId=chunk.metadata.get('doc_id')
    user_prompt= f"Redacta una pregunta simple sobre el contenido del siguiente texto. La pregunta se utilizara para evaluar la retrieval de un vectorstore. Solo genera la pregunta.  Texto: {chunkContent}"
    messages = [HumanMessage(content=user_prompt)]
    model_output = llm.invoke(messages)    
    final_result = model_output.content.strip()
    #print("El texto recibido fue: \n"+chunkContent)
    #print(model_output["choices"][0]["text"])
    print(f"pedido: {user_prompt} \n --- \n respuesta: {final_result}")
    evaluation_dataset_AIgenerated.append({"question":f"{final_result}","ground_truth_doc_id":f"{chunkId}"})

pedido: Redacta una pregunta simple sobre el contenido del siguiente texto. La pregunta se utilizara para evaluar la retrieval de un vectorstore. Solo genera la pregunta.  Texto: Un problema de **Valores y Vectores Propios** consiste en encontrar los vectores \(v\) tales que son direcciones invariantes de la transformación lineal dada por la matriz \(A\) (NxN). Esto se expresa como: \(A v = \lambda v \) siendo \(\lambda\) el escalar que cambia el módulo del vector cuya dirección permanece invariante. Se denomina **autovalor** \( \lambda \) y **autovector** \(v\). El sistema de ecuaciones puede escribirse de la forma: \((A - \lambda I) v = 0 \) donde interesan las soluciones \(v\) distintas de la trivial,\(v = 0\). Esto está garantizado sí y sólo sí \(det(A - \lambda I) = 0\). El determinante constituye un polinomio de grado N en el autovalor \(\lambda \), y se denomina **polinomio característico**. Las raíces de dicho polinomio son los autovalores \(\lambda \) de la matriz \(A\) para l

pedido: Redacta una pregunta simple sobre el contenido del siguiente texto. La pregunta se utilizara para evaluar la retrieval de un vectorstore. Solo genera la pregunta.  Texto: El objetivo principal es encontrar la **integral definida** `I` en `R`, dada por `I = Integral from X0 to Xn of f(x) dx`.
Basándose en la relación `Integral(f(x)dx) = Integral(Pn(x)dx) + Integral(En(x)dx)`, la integral definida `I` se evalúa como la suma:
`I = In + En`
Donde `In` se denomina **cuadratura** y `En` es el **error de truncamiento**.  
Todos los métodos de integración numérica comparten la estructura en la que la cuadratura `In` se expresa como una suma:
`In = sum(wj * f(xj))`
Los **coeficientes `wj`** se determinan según cada regla. El **orden de la regla de cuadratura** se define como el máximo grado del polinomio que dicha regla integra de forma exacta, es decir, para el cual `En = 0`.  
Existen dos tipos principales de cuadratura: 
 --- 
 respuesta: ¿Cuál es la relación entre la integral defini

pedido: Redacta una pregunta simple sobre el contenido del siguiente texto. La pregunta se utilizara para evaluar la retrieval de un vectorstore. Solo genera la pregunta.  Texto: La **regla de Simpson** es una regla de **orden 3**, lo que significa que integra de forma exacta polinomios de grado hasta 3. Permite aproximar la integral `I` mediante la fórmula:
`I = h/3 * (f(x0) + 4*f(x1) + f(x2)) - h^5/90 * f^(4)(xi)`
Donde `xi` es un valor entre `x0` y `x2`. 
 --- 
 respuesta: ¿Qué tipo de polinomios integra exactamente la regla de Simpson?
pedido: Redacta una pregunta simple sobre el contenido del siguiente texto. La pregunta se utilizara para evaluar la retrieval de un vectorstore. Solo genera la pregunta.  Texto: Es una cuadratura de Newton-Cotes con `n=2`, es decir, utiliza **tres puntos**. Se interpola la función con un polinomio de Lagrange de grado dos y se integra este polinomio de forma aproximada.
Si los intervalos son iguales (`h1 = h2 = h`), la fórmula de Simpson es:
`I = h/

In [12]:
evaluation_dataset_AIgenerated

[{'question': '¿Qué se denomina autovalor y autovector en un problema de valores y vectores propios?',
  'ground_truth_doc_id': 'chunk_0'},
 {'question': '¿Cuál es el autovector al que tienden los vectores obtenidos al premultiplicar repetidamente un vector por la matriz A?',
  'ground_truth_doc_id': 'chunk_1'},
 {'question': '¿Cuál es el paso inicial en el algoritmo descrito?',
  'ground_truth_doc_id': 'chunk_2'},
 {'question': '¿Qué autovalor de la matriz A se encuentra utilizando el Método de la Potencia aplicado a la matriz inversa A⁻¹?',
  'ground_truth_doc_id': 'chunk_3'},
 {'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
  'ground_truth_doc_id': 'chunk_4'},
 {'question': '¿Qué se puede hacer para calcular la integral de una función cuando se conoce de forma discreta?',
  'ground_truth_doc_id': 'chunk_5'},
 {'question': '¿Cuál es la relación entre la integral definida, la cuadratura y el error de truncamiento según el tex

# TESTING Embeddings

### Testing vectorstore as retriever

Process: Given a question, a list "k" of chunks is retrieved.
#### Definiciones
- hit: the ground truth ID is in the retrieved IDs
- rank: position of the correct chunk in the retrieved chunks.
- reciprocal_ranks = (1 / rank)

#### Medidas
- hit rate = (hits from all questions / total_questions) * 100
- mrr=mean(reciprocal_ranks from all questions)

In [13]:
def evaluate_vectorstore_as_retriever(eval_dataset, vector_store, k=5):
    """
    Evaluates the performance of a retriever using a given dataset.

    Args:
        eval_dataset (list): A list of dictionaries with "question" and "ground_truth_doc_id".
        vector_store: The ChromaDB vector store instance.
        k (int): The number of top documents to retrieve for evaluation.

    Returns:
        dict: A dictionary containing the calculated metrics.
    """
    hits = 0
    reciprocal_ranks = []
    misses = [] # To store information about failed queries for later analysis

    print(f"Starting evaluation for k={k}...")

    for item in eval_dataset:
        question = item["question"]
        ground_truth_id = item["ground_truth_doc_id"]
        
        # Perform the similarity search
        # The result is a list of tuples: [(Document, score), (Document, score), ...]
        retrieved_docs_with_scores = vector_store.similarity_search_with_score(question, k=k)
        
        # Extract the doc_ids from the metadata of the retrieved documents
        retrieved_ids = [doc.metadata.get('doc_id') for doc, score in retrieved_docs_with_scores]
        
        # Check if the ground truth ID is in the retrieved IDs
        if ground_truth_id in retrieved_ids:
            hits += 1
            # Find the rank (position) of the correct document. Ranks are 1-based.
            rank = retrieved_ids.index(ground_truth_id) + 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)
            misses.append({
                "question": question,
                "expected": ground_truth_id,
                "retrieved": retrieved_ids
            })

    total_questions = len(eval_dataset)
    hit_rate = (hits / total_questions) * 100
    mrr = np.mean(reciprocal_ranks)

    return {
        "hit_rate_at_k": k,
        "hit_rate": f"{hit_rate:.2f}%",
        "mrr": f"{mrr:.4f}",
        "total_questions": total_questions,
        "hits": hits,
        "misses_count": len(misses),
        "misses": misses
    }

In [14]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore, k=5)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=5...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 5,
 'hit_rate': '78.57%',
 'mrr': '0.5952',
 'total_questions': 28,
 'hits': 22,
 'misses_count': 6,
 'misses': [{'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_2', 'chunk_6', 'chunk_3', 'chunk_0', 'chunk_1']},
  {'question': '¿Qué método se utiliza para interpolar una función dada por n+1 puntos?',
   'expected': 'chunk_9',
   'retrieved': ['chunk_11', 'chunk_18', 'chunk_23', 'chunk_22', 'chunk_6']},
  {'question': '¿Cuál es el orden del error en la regla de los trapecios?',
   'expected': 'chunk_12',
   'retrieved': ['chunk_13', 'chunk_20', 'chunk_10', 'chunk_23', 'chunk_24']},
  {'question': '¿Cuáles son los valores de `a0` y `a1` que se determinan en el método descrito?',
   'expected': 'chunk_14',
   'retrieved': ['chunk_22', 'chunk_23', 'chunk_6', 'chunk_2', 'chunk_11']},
  {'question': '¿Cuál es el orden del error para la regla de trapecios múltiple?',


In [15]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore, k=10)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=10...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 10,
 'hit_rate': '78.57%',
 'mrr': '0.5952',
 'total_questions': 28,
 'hits': 22,
 'misses_count': 6,
 'misses': [{'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_2',
    'chunk_6',
    'chunk_3',
    'chunk_0',
    'chunk_1',
    'chunk_22',
    'chunk_18',
    'chunk_27',
    'chunk_13',
    'chunk_11']},
  {'question': '¿Qué método se utiliza para interpolar una función dada por n+1 puntos?',
   'expected': 'chunk_9',
   'retrieved': ['chunk_11',
    'chunk_18',
    'chunk_23',
    'chunk_22',
    'chunk_6',
    'chunk_5',
    'chunk_2',
    'chunk_24',
    'chunk_27',
    'chunk_13']},
  {'question': '¿Cuál es el orden del error en la regla de los trapecios?',
   'expected': 'chunk_12',
   'retrieved': ['chunk_13',
    'chunk_20',
    'chunk_10',
    'chunk_23',
    'chunk_24',
    'chunk_2',
    'chunk_6',
    'chunk_18',
    'chunk_11',
    'chunk_22']},


In [32]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_allminilm, k=5)

init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=5...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 5,
 'hit_rate': '78.57%',
 'mrr': '0.5667',
 'total_questions': 28,
 'hits': 22,
 'misses_count': 6,
 'misses': [{'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_5', 'chunk_5', 'chunk_7', 'chunk_7', 'chunk_24']},
  {'question': '¿Cuál es el orden del error en la regla de los trapecios?',
   'expected': 'chunk_12',
   'retrieved': ['chunk_10', 'chunk_10', 'chunk_20', 'chunk_20', 'chunk_24']},
  {'question': '¿Cuál es la complejidad del error en la regla de trapecios según el texto?',
   'expected': 'chunk_13',
   'retrieved': ['chunk_20', 'chunk_20', 'chunk_10', 'chunk_10', 'chunk_24']},
  {'question': '¿Cuál es el orden del error para la regla de trapecios múltiple?',
   'expected': 'chunk_15',
   'retrieved': ['chunk_10', 'chunk_10', 'chunk_20', 'chunk_20', 'chunk_24']},
  {'question': '¿Cuáles son los valores de los coeficientes C-1, C0 y C1?',
   'expected':

In [27]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_allminilm, k=10)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=10...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 10,
 'hit_rate': '85.71%',
 'mrr': '0.5769',
 'total_questions': 28,
 'hits': 24,
 'misses_count': 4,
 'misses': [{'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_5',
    'chunk_5',
    'chunk_7',
    'chunk_7',
    'chunk_24',
    'chunk_24',
    'chunk_2',
    'chunk_2',
    'chunk_9',
    'chunk_9']},
  {'question': '¿Cuál es el orden del error en la regla de los trapecios?',
   'expected': 'chunk_12',
   'retrieved': ['chunk_10',
    'chunk_10',
    'chunk_20',
    'chunk_20',
    'chunk_24',
    'chunk_24',
    'chunk_2',
    'chunk_2',
    'chunk_23',
    'chunk_23']},
  {'question': '¿Cuál es la complejidad del error en la regla de trapecios según el texto?',
   'expected': 'chunk_13',
   'retrieved': ['chunk_20',
    'chunk_20',
    'chunk_10',
    'chunk_10',
    'chunk_24',
    'chunk_24',
    'chunk_2',
    'chunk_2',
    'chunk_7',
    'chunk_7']},


In [30]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_allminilm, k=15)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=15...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 15,
 'hit_rate': '89.29%',
 'mrr': '0.5796',
 'total_questions': 28,
 'hits': 25,
 'misses_count': 3,
 'misses': [{'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_5',
    'chunk_5',
    'chunk_7',
    'chunk_7',
    'chunk_24',
    'chunk_24',
    'chunk_2',
    'chunk_2',
    'chunk_9',
    'chunk_9',
    'chunk_8',
    'chunk_8',
    'chunk_20',
    'chunk_20',
    'chunk_27']},
  {'question': '¿Cuál es el orden del error en la regla de los trapecios?',
   'expected': 'chunk_12',
   'retrieved': ['chunk_10',
    'chunk_10',
    'chunk_20',
    'chunk_20',
    'chunk_24',
    'chunk_24',
    'chunk_2',
    'chunk_2',
    'chunk_23',
    'chunk_23',
    'chunk_16',
    'chunk_16',
    'chunk_7',
    'chunk_7',
    'chunk_26']},
  {'question': '¿Cuál es el orden del error para la regla de trapecios múltiple?',
   'expected': 'chunk_15',
   'retrieved': ['chunk_1

In [33]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_nomic, k=5)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=5...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 5,
 'hit_rate': '89.29%',
 'mrr': '0.6994',
 'total_questions': 28,
 'hits': 25,
 'misses_count': 3,
 'misses': [{'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_1', 'chunk_3', 'chunk_23', 'chunk_26', 'chunk_24']},
  {'question': '¿Cuál es el orden del error para la regla de trapecios múltiple?',
   'expected': 'chunk_15',
   'retrieved': ['chunk_10', 'chunk_23', 'chunk_16', 'chunk_20', 'chunk_24']},
  {'question': '¿Cuáles son los valores de los coeficientes C-1, C0 y C1?',
   'expected': 'chunk_19',
   'retrieved': ['chunk_8', 'chunk_7', 'chunk_23', 'chunk_0', 'chunk_14']}]}

In [28]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_nomic, k=10)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=10...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 10,
 'hit_rate': '92.86%',
 'mrr': '0.7054',
 'total_questions': 28,
 'hits': 26,
 'misses_count': 2,
 'misses': [{'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_1',
    'chunk_3',
    'chunk_23',
    'chunk_26',
    'chunk_24',
    'chunk_19',
    'chunk_16',
    'chunk_10',
    'chunk_21',
    'chunk_0']},
  {'question': '¿Cuál es el orden del error para la regla de trapecios múltiple?',
   'expected': 'chunk_15',
   'retrieved': ['chunk_10',
    'chunk_23',
    'chunk_16',
    'chunk_20',
    'chunk_24',
    'chunk_13',
    'chunk_12',
    'chunk_26',
    'chunk_14',
    'chunk_2']}]}

In [34]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_nomic, k=15)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=15...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 15,
 'hit_rate': '92.86%',
 'mrr': '0.7054',
 'total_questions': 28,
 'hits': 26,
 'misses_count': 2,
 'misses': [{'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_1',
    'chunk_3',
    'chunk_23',
    'chunk_26',
    'chunk_24',
    'chunk_19',
    'chunk_16',
    'chunk_10',
    'chunk_21',
    'chunk_0',
    'chunk_14',
    'chunk_11',
    'chunk_9',
    'chunk_27',
    'chunk_6']},
  {'question': '¿Cuál es el orden del error para la regla de trapecios múltiple?',
   'expected': 'chunk_15',
   'retrieved': ['chunk_10',
    'chunk_23',
    'chunk_16',
    'chunk_20',
    'chunk_24',
    'chunk_13',
    'chunk_12',
    'chunk_26',
    'chunk_14',
    'chunk_2',
    'chunk_21',
    'chunk_11',
    'chunk_17',
    'chunk_22',
    'chunk_1']}]}

In [35]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_bge_m3_q8, k=5)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=5...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 5,
 'hit_rate': '78.57%',
 'mrr': '0.5952',
 'total_questions': 28,
 'hits': 22,
 'misses_count': 6,
 'misses': [{'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_2', 'chunk_6', 'chunk_3', 'chunk_0', 'chunk_1']},
  {'question': '¿Qué método se utiliza para interpolar una función dada por n+1 puntos?',
   'expected': 'chunk_9',
   'retrieved': ['chunk_11', 'chunk_18', 'chunk_23', 'chunk_22', 'chunk_6']},
  {'question': '¿Cuál es el orden del error en la regla de los trapecios?',
   'expected': 'chunk_12',
   'retrieved': ['chunk_13', 'chunk_20', 'chunk_10', 'chunk_23', 'chunk_24']},
  {'question': '¿Cuáles son los valores de `a0` y `a1` que se determinan en el método descrito?',
   'expected': 'chunk_14',
   'retrieved': ['chunk_22', 'chunk_23', 'chunk_6', 'chunk_2', 'chunk_11']},
  {'question': '¿Cuál es el orden del error para la regla de trapecios múltiple?',


In [29]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_bge_m3_q8, k=10)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=10...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 10,
 'hit_rate': '78.57%',
 'mrr': '0.5952',
 'total_questions': 28,
 'hits': 22,
 'misses_count': 6,
 'misses': [{'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_2',
    'chunk_6',
    'chunk_3',
    'chunk_0',
    'chunk_1',
    'chunk_22',
    'chunk_18',
    'chunk_27',
    'chunk_13',
    'chunk_11']},
  {'question': '¿Qué método se utiliza para interpolar una función dada por n+1 puntos?',
   'expected': 'chunk_9',
   'retrieved': ['chunk_11',
    'chunk_18',
    'chunk_23',
    'chunk_22',
    'chunk_6',
    'chunk_5',
    'chunk_2',
    'chunk_24',
    'chunk_27',
    'chunk_13']},
  {'question': '¿Cuál es el orden del error en la regla de los trapecios?',
   'expected': 'chunk_12',
   'retrieved': ['chunk_13',
    'chunk_20',
    'chunk_10',
    'chunk_23',
    'chunk_24',
    'chunk_2',
    'chunk_6',
    'chunk_18',
    'chunk_11',
    'chunk_22']},


In [31]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_bge_m3_q8, k=15)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=15...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 15,
 'hit_rate': '82.14%',
 'mrr': '0.5980',
 'total_questions': 28,
 'hits': 23,
 'misses_count': 5,
 'misses': [{'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_2',
    'chunk_6',
    'chunk_3',
    'chunk_0',
    'chunk_1',
    'chunk_22',
    'chunk_18',
    'chunk_27',
    'chunk_13',
    'chunk_11',
    'chunk_23',
    'chunk_7',
    'chunk_20',
    'chunk_26',
    'chunk_5']},
  {'question': '¿Qué método se utiliza para interpolar una función dada por n+1 puntos?',
   'expected': 'chunk_9',
   'retrieved': ['chunk_11',
    'chunk_18',
    'chunk_23',
    'chunk_22',
    'chunk_6',
    'chunk_5',
    'chunk_2',
    'chunk_24',
    'chunk_27',
    'chunk_13',
    'chunk_0',
    'chunk_3',
    'chunk_26',
    'chunk_8',
    'chunk_20']},
  {'question': '¿Cuál es el orden del error en la regla de los trapecios?',
   'expected': 'chunk_12',
   'retrieved': ['

In [39]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_nomic_moe, k=5)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=5...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 5,
 'hit_rate': '89.29%',
 'mrr': '0.7304',
 'total_questions': 28,
 'hits': 25,
 'misses_count': 3,
 'misses': [{'question': '¿Cómo se modifica la matriz A en el proceso iterativo para encontrar los modos intermedios?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_1', 'chunk_3', 'chunk_0', 'chunk_2', 'chunk_14']},
  {'question': '¿Cuál es la relación entre la integral definida, la cuadratura y el error de truncamiento según el texto?',
   'expected': 'chunk_6',
   'retrieved': ['chunk_24', 'chunk_20', 'chunk_22', 'chunk_12', 'chunk_18']},
  {'question': '¿Cuál es el orden del error para la regla de trapecios múltiple?',
   'expected': 'chunk_15',
   'retrieved': ['chunk_10', 'chunk_13', 'chunk_16', 'chunk_12', 'chunk_24']}]}

In [40]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_nomic_moe, k=10)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=10...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 10,
 'hit_rate': '96.43%',
 'mrr': '0.7390',
 'total_questions': 28,
 'hits': 27,
 'misses_count': 1,
 'misses': [{'question': '¿Cuál es el orden del error para la regla de trapecios múltiple?',
   'expected': 'chunk_15',
   'retrieved': ['chunk_10',
    'chunk_13',
    'chunk_16',
    'chunk_12',
    'chunk_24',
    'chunk_20',
    'chunk_23',
    'chunk_26',
    'chunk_14',
    'chunk_27']}]}

In [41]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_nomic_moe, k=15)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=15...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 15,
 'hit_rate': '96.43%',
 'mrr': '0.7390',
 'total_questions': 28,
 'hits': 27,
 'misses_count': 1,
 'misses': [{'question': '¿Cuál es el orden del error para la regla de trapecios múltiple?',
   'expected': 'chunk_15',
   'retrieved': ['chunk_10',
    'chunk_13',
    'chunk_16',
    'chunk_12',
    'chunk_24',
    'chunk_20',
    'chunk_23',
    'chunk_26',
    'chunk_14',
    'chunk_27',
    'chunk_11',
    'chunk_17',
    'chunk_22',
    'chunk_18',
    'chunk_4']}]}

In [45]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_mxbai, k=5)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=5...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 5,
 'hit_rate': '71.43%',
 'mrr': '0.4054',
 'total_questions': 28,
 'hits': 20,
 'misses_count': 8,
 'misses': [{'question': '¿Qué se puede hacer para calcular la integral de una función cuando se conoce de forma discreta?',
   'expected': 'chunk_5',
   'retrieved': ['chunk_12', 'chunk_19', 'chunk_15', 'chunk_9', 'chunk_8']},
  {'question': '¿Para qué tipo de polinomios la regla de los trapecios es exacta?',
   'expected': 'chunk_10',
   'retrieved': ['chunk_9', 'chunk_13', 'chunk_16', 'chunk_23', 'chunk_24']},
  {'question': '¿Cuál es el orden del error en la regla de los trapecios?',
   'expected': 'chunk_12',
   'retrieved': ['chunk_13', 'chunk_23', 'chunk_2', 'chunk_16', 'chunk_20']},
  {'question': '¿Cuáles son los valores de `a0` y `a1` que se determinan en el método descrito?',
   'expected': 'chunk_14',
   'retrieved': ['chunk_8', 'chunk_23', 'chunk_12', 'chunk_2', 'chunk_15']},
  {'question': '¿Cuál es el orden del error para la regla de trapecios múltiple?'

In [46]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_mxbai, k=10)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=10...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

{'hit_rate_at_k': 10,
 'hit_rate': '89.29%',
 'mrr': '0.4299',
 'total_questions': 28,
 'hits': 25,
 'misses_count': 3,
 'misses': [{'question': '¿Cuáles son los valores de `a0` y `a1` que se determinan en el método descrito?',
   'expected': 'chunk_14',
   'retrieved': ['chunk_8',
    'chunk_23',
    'chunk_12',
    'chunk_2',
    'chunk_15',
    'chunk_7',
    'chunk_3',
    'chunk_4',
    'chunk_19',
    'chunk_1']},
  {'question': '¿Cuál es el orden del error para la regla de trapecios múltiple?',
   'expected': 'chunk_15',
   'retrieved': ['chunk_13',
    'chunk_23',
    'chunk_2',
    'chunk_16',
    'chunk_24',
    'chunk_12',
    'chunk_20',
    'chunk_26',
    'chunk_9',
    'chunk_17']},
  {'question': '¿Cuál es el objetivo principal del algoritmo de integración de Romberg descrito en el texto?',
   'expected': 'chunk_27',
   'retrieved': ['chunk_3',
    'chunk_4',
    'chunk_8',
    'chunk_23',
    'chunk_1',
    'chunk_6',
    'chunk_26',
    'chunk_0',
    'chunk_12',
    

In [ ]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore_mxbai, k=15)

init: embeddings required but some input tokens were not marked as outputs -> overriding


Starting evaluation for k=15...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

### Testing de Re-ranker

In [23]:
# Creamos el retriever base. Este será "envuelto" por el compresor.
# Le damos un 'k' más alto porque esperamos que el filtro descarte algunos resultados.
base_retriever = vectorStore.as_retriever(
    search_type="similarity_score_threshold", 
    search_kwargs={"score_threshold": 0.1,"k": 10}
    #search_type="similarity", # 'similarity' with a 'k' is often more reliable
    #search_kwargs={"k": 10} # Retrieve more documents to give the reranker more to work with
)
# ¡IMPORTANTE! Usamos la MISMA instancia 'embedding_encoder' que para la DB.
redundant_filter = EmbeddingsRedundantFilter(embeddings=embedding_encoder)
reranker = CrossEncoderReranker(model=retrieval_reranker, top_n=3)
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[redundant_filter, reranker]
)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor, 
    base_retriever=base_retriever
)

In [30]:
def evaluate_retriever(
    retriever: Any,
    eval_dataset: List[Dict[str, str]],
    retriever_name: str
) -> Dict[str, Any]:
    """
    Evaluates the performance of any retriever with a .invoke() method.

    Args:
        retriever (Any): A retriever object that has an .invoke(query) method
                         which returns a list of Document objects.
        eval_dataset (list): A list of dictionaries with "question" and 
                             "ground_truth_doc_id".
        retriever_name (str): A descriptive name for the retriever being tested
                              (e.g., "Base Retriever k=5").

    Returns:
        dict: A dictionary containing the calculated metrics and a list of misses.
    """
    hits = 0
    reciprocal_ranks = []
    misses = []  # To store information about failed queries for analysis

    print(f"--- Starting evaluation for: {retriever_name} ---")

    for item in eval_dataset:
        question = item["question"]
        ground_truth_id = item["ground_truth_doc_id"]
        
        # 1. Use the generic .invoke() method
        retrieved_docs = retriever.invoke(question)
        
        # 2. Extract doc_ids from the list of Document objects
        retrieved_ids = [doc.metadata.get('doc_id') for doc in retrieved_docs]
        
        # 3. Perform the evaluation logic (this part remains the same)
        if ground_truth_id in retrieved_ids:
            hits += 1
            # Ranks are 1-based
            rank = retrieved_ids.index(ground_truth_id) + 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)
            misses.append({
                "question": question,
                "expected": ground_truth_id,
                "retrieved": retrieved_ids
            })

    total_questions = len(eval_dataset)
    hit_rate = (hits / total_questions) * 100
    mrr = np.mean(reciprocal_ranks) if reciprocal_ranks else 0

    print(f"Evaluation finished for: {retriever_name}\n")

    return {
        "retriever_name": retriever_name,
        "hit_rate": f"{hit_rate:.2f}%",
        "mrr": f"{mrr:.4f}",
        "total_questions": total_questions,
        "hits": hits,
        "misses_count": len(misses),
        "misses": misses
    }

def print_results(results: Dict[str, Any]):
    """Helper function to print evaluation results in a readable format."""
    print(f"--- Retrieval Evaluation Results for: {results['retriever_name']} ---")
    print(f"Hit Rate: {results['hit_rate']}")
    print(f"Mean Reciprocal Rank (MRR): {results['mrr']}")
    print(f"Correctly Retrieved (Hits): {results['hits']} / {results['total_questions']}")
    print("-" * 50)

    if results['misses_count'] > 0:
        print(f"\nAnalysis of {results['misses_count']} Misses:")
        # Print details for the first 3 misses for brevity
        for i, miss in enumerate(results['misses'][:3]):
            print(f"\nMiss {i+1}:")
            print(f"  Question: '{miss['question']}'")
            print(f"  Expected Doc ID: {miss['expected']}")
            print(f"  Retrieved IDs:   {miss['retrieved']}")
        if results['misses_count'] > 3:
            print("\n(And more...)")
    print("\n")

In [ ]:
# Evaluate the base retriever
base_retriever_results = evaluate_retriever(
    retriever=base_retriever,
    eval_dataset=evaluation_dataset_AIgenerated,
    retriever_name="Base Retriever"
)
# Evaluate the compression retriever
compression_retriever_results = evaluate_retriever(
    retriever=compression_retriever,
    eval_dataset=evaluation_dataset_AIgenerated,
    retriever_name="Compression Retriever"
)

# --- Reporting Phase ---
print_results(base_retriever_results)
print_results(compression_retriever_results)

--- Starting evaluation for: Base Retriever ---
Evaluation finished for: Base Retriever

--- Starting evaluation for: Compression Retriever ---


In [36]:
print_results(base_retriever_results)

--- Retrieval Evaluation Results for: Base Retriever ---
Hit Rate: 95.00%
Mean Reciprocal Rank (MRR): 0.7083
Correctly Retrieved (Hits): 19 / 20
--------------------------------------------------

Analysis of 1 Misses:

Miss 1:
  Question: '¿Cuál es el pseudocodigo del método de la potencia?'
  Expected Doc ID: chunk_4
  Retrieved IDs:   ['chunk_19', 'chunk_30', 'chunk_5', 'chunk_12', 'chunk_17', 'chunk_21', 'chunk_2', 'chunk_14', 'chunk_28', 'chunk_27']




In [21]:
def DefineGemmaPrompt():
    """
    Defines the prompt template for Gemma 3 models via Google API.
    
    Since Gemma 3 does not support system prompts through this API, this function
    combines the system instructions and the user query into a single human/user
    message template.
    """
    system_instructions = """Eres un experto en pedagogía para estudiantes universitarios de la generación Z y profesor de la cátedra de Métodos Numéricos en la Facultad de Ingeniería. Tu objetivo es guiar al usuario a lograr una comprensión más profunda sobre su pregunta.

Recibirás una PREGUNTA y un CONTEXTO de las notas de clase. Sigue estas reglas estrictamente:
1. Responde a la PREGUNTA utilizando ÚNICAMENTE el CONTEXTO proporcionado. No uses información de otras fuentes. Si no hay CONTEXTO, indica que la respuesta no ha sido encontrada.
2. Formatea siempre tus respuestas utilizando Markdown para mejorar la legibilidad.
3. Después de tu explicación, incluye una sección de TAREAS ACCIONABLES o PREGUNTAS DE REFLEXIÓN para que el estudiante aplique o profundice su conocimiento.
4. Responde siempre en español. Sé útil y claro."""

    # Combine the instructions and the dynamic parts into a single template string.
    # The model will treat the entire block as the user's input.
    full_prompt_string = (
        f"{system_instructions}\n\n"
        "--- \n\n"  # Using a separator can sometimes help the model distinguish instructions from data.
        "CONTEXTO:\n{context}\n\n"
        "PREGUNTA:\n{question}"
    )

    # Create the template from a single string. LangChain will treat this
    # as a single "human" message by default in many chains.
    prompt_template = ChatPromptTemplate.from_template(full_prompt_string)
    
    return prompt_template

def ProcessInput(question,retriever,llm):
    #Data pipeline: user query->retrieve chunks->join them->inject in prompt-> get LLM response
    prompt = DefineGemmaPrompt()
    rag_chain = (
        {"context": retriever | join_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain.invoke(question)

In [19]:
#llm=LoadOllamaLLM()
#llm=LoadGoogleLLM()
llm=LoadLLMFromllamaCPP()

In [24]:
#Uso de herramienta
query="Que pasa si ingreso un autovector como vector inicial en el metodo de la potencia?"

response=ProcessInput(query,base_retriever,llm)
print(response)

De acuerdo al contexto proporcionado, si ingresas un autovector como vector inicial en el método de la potencia, el proceso se simplifica significativamente. 

El método de la potencia, como se explica, converge al autovector dominante (el asociado al autovalor de mayor valor absoluto) a través de sucesivas premultiplicaciones de un vector inicial por la matriz `A`.  Si el vector inicial `x` ya es un autovector `v_1` asociado al autovalor dominante `λ_1`, entonces:

`A x = A v_1 = λ_1 v_1 = λ_1 x`

Cada iteración del método simplemente multiplicará el vector `x` por el autovalor `λ_1`.  El vector permanecerá en la dirección del autovector `v_1` y el cociente `x_{k+1}(j) / x_k(j)` convergerá inmediatamente a `λ_1` sin necesidad de múltiples iteraciones.  El escalamiento (normalización) seguirá siendo necesario para evitar problemas de overflow o underflow, pero la convergencia será mucho más rápida y directa.

---

**TAREAS ACCIONABLES / PREGUNTAS DE REFLEXIÓN:**

1.  **Considera una ma